In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 勞動基準法qa百問百答第二版.pdf to 勞動基準法qa百問百答第二版 (7).pdf


In [ ]:
import os
!pip install langchain-chroma
!pip install langchain-openai
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

os.environ["OPENAI_API_KEY"] = "XXX" # 請自行放上自己的OpenAI API !!!

embeddings = OpenAIEmbeddings()

In [ ]:
!pip install -q pypdf langchain-community

from langchain_community.document_loaders import PyPDFLoader

# PDF 檔案路徑
pdf_path = "/content/勞動基準法qa百問百答第二版.pdf"

# 載入 PDF
loader = PyPDFLoader(pdf_path)

docs_list = loader.load()

print(f"成功載入 {len(docs_list)} 頁")

成功載入 161 頁


In [ ]:
!pip install PyMuPDF
import fitz
from langchain_core.documents import Document

# 開啟 PDF
doc = fitz.open(pdf_path)

page_documents = []

for page_index, page in enumerate(doc):

    text = page.get_text().strip()

    # 空白頁不加入
    if not text:
        continue

    page_documents.append(
        Document(
            page_content=text,
            metadata={
                "source": "勞動基準法QA百問百答第二版",
                "page": page_index + 1
            }
        )
    )

print("PDF 總頁數：", len(doc))
print("成功建立 Page Documents：", len(page_documents))

PDF 總頁數： 161
成功建立 Page Documents： 159


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "。", "，", " ", ""]
)

chunks = text_splitter.split_documents(page_documents)

print("Page Documents：", len(page_documents))
print("切分後 Chunks：", len(chunks))

Page Documents： 159
切分後 Chunks： 168


In [ ]:
import chromadb
from langchain_chroma import Chroma

# Get or create a Chroma client. Since no persist_directory is specified,
# this implicitly connects to the in-memory Chroma instance.
chroma_client = chromadb.Client()

collection_name = "labor_law_rag_final"

try:
    # Attempt to delete the collection if it already exists from a previous run
    chroma_client.delete_collection(name=collection_name)
    print(f"Successfully deleted existing Chroma collection: '{collection_name}'.")
except Exception as e:
    # This exception will occur if the collection doesn't exist, which is fine.
    print(f"Collection '{collection_name}' not found or could not be deleted (might not exist yet): {e}")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=collection_name,
    client=chroma_client # Pass the explicit client to ensure it's used
)

print("Vector DB 建立完成")
print("Chunk 數量：", len(chunks))

Successfully deleted existing Chroma collection: 'labor_law_rag_final'.
Vector DB 建立完成
Chunk 數量： 168


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever 建立完成！")

Retriever 建立完成！


In [ ]:
evaluation_dataset = [
    {
        "id": "E01",
        "question": "我們在宮廟裡當專職抬轎和辦公的員工，每天固定打卡上工、領月薪，也會有勞基法特休或退休金的保障嗎？還是宗教團體就不算？",
        "type": "高難度情境型",
        "ground_truth_pages": [14] # 誘騙重點：向量模型會被「宮廟、宗教團體」吸引去抓不適用的特殊條款，但核心在於「固定打卡領薪的僱傭關係」。
    },
    {
        "id": "E02",
        "question": "我是被A公司面試錄取的，但他們把我派到B公司去當總務，我的打卡和主管都是B公司的，那如果我被資遣，到底是要跟哪一家公司要非自願離職證明？",
        "type": "長文本陷阱型",
        "ground_truth_pages": [17] # 誘騙重點：同時出現兩家公司和「資遣證明」，會引誘第一階段抓到一般資遣規定，但核心是「派遣勞工權利歸屬」。
    },
    {
        "id": "E03",
        "question": "老闆說因為行業性質特殊，所以每年年底都要跟我重新簽一次定期契約。今年約滿了他直接叫我不用來了，還說這是合約自然到期所以不用付資遣費，這在法律上站得住腳嗎？",
        "type": "偽裝型法規題",
        "ground_truth_pages": [21] # 誘騙重點：大量出現「定期契約、合約到期」，容易讓向量誤判為合法的定期契約期滿，但核心是「非固定工作偽造定期契約與資遣費」。
    },
    {
        "id": "E04",
        "question": "一早進公司打完卡，主管突然叫我進辦公室，拿出一張單子說業績不好叫我今天辦交接直接做到下班。遇到這種毫無預警的當天開除，我手頭上可以跟公司據理力爭哪些權益和補助？",
        "type": "情緒與口語干擾型",
        "ground_truth_pages": [25] # 誘騙重點：「開除、交接、業績不好」干擾字極多，核心在於「當天無預警資遣時的法定保障（預告工資與資遣費）」。
    },
    {
        "id": "E05",
        "question": "公司昨天無預警資遣我，完全沒提前講，主管說他查過法律了，只要在最後結算薪水時，多塞給我5天的薪水當作補償就完全合法、不用提早通知，這樣做真的可以嗎？",
        "type": "數字與反向陷阱型",
        "ground_truth_pages": [27] # 誘騙重點：出現「多給5天薪水、合法、不用通知」，考驗模型對於「資遣預告期天數與預告工資折算」的精準法規對齊。
    },
    {
        "id": "E06",
        "question": "我上個禮拜就遞交了辭職信，但老闆死活不肯簽字，甚至威脅我說他不准假、私自離職要告我毀損商業利益。如果合約規定的離職預告期已經到了，我能不理他直接不去上班嗎？",
        "type": "強干擾情境型",
        "ground_truth_pages": [30] # 誘騙重點：出現「告我、不准假、威脅、毀損利益」，會誘導向量走向法律訴訟，但核心是「勞工自願離職不需雇主同意之形成權」。
    },
    {
        "id": "E07",
        "question": "因為業務緊縮我被裁員了，公司雖然答應會付資遣費，但不管我怎麼求，他們就是不願意開非自願離職證明書給我，說怕被政府查。沒有這張紙我根本無法申請失業給付，我該怎麼爭取？",
        "type": "多重名詞干擾型",
        "ground_truth_pages": [32] # 誘騙重點：「資遣費、裁員、失業給付、政府查」四合一，考驗重排模型定位於「雇主拒開非自願離職證明之救濟」。
    },
    {
        "id": "E08",
        "question": "上個月因為請了兩天病假，結果全勤獎金被扣光，導致最後銀行帳戶入帳的總薪資扣一扣居然比政府規定的法定基本工資還要低！老闆說是因為我沒拿到全勤，這樣真的沒違法嗎？",
        "type": "高鑑別度計算法規型",
        "ground_truth_pages": [34] # 誘騙重點：「請病假、扣全勤獎金」是非常強的干擾，容易引導至請假扣薪規定，但底線核心是「全勤獎金與基本工資的關係」。
    },
    {
        "id": "E09",
        "question": "上禮拜在倉庫搬運不小心摔碎了一批貨物，經理氣炸了，直接甩出一張切結書，說要從我每個月的月薪裡面直接扣除五千元當作賠償，直到扣完為止，公司有權力做這種單方面的強制扣款嗎？",
        "type": "生活情境陷阱型",
        "ground_truth_pages": [40] # 誘騙重點：「摔碎貨物、切結書、賠償」會吸引向量去抓民事損害賠償，但核心是勞基法的「薪資全額直接給付原則（不得逕自扣發薪資）」。
    },
    {
        "id": "E10",
        "question": "公司這個月財務週轉不靈發不出薪水，老闆居然載了一整箱我們公司生產的保養品過來，跟我說這些商品市值大於我的薪水，要拿這箱貨抵我上個月的工資，我可以義正嚴辭拒絕並檢舉他嗎？",
        "type": "高度口語隱喻型",
        "ground_truth_pages": [50] # 誘騙重點：口語的「拿貨抵工資、財務週轉不靈」，極度考驗模型能否轉換為法規核心「工資應以法定貨幣給付原則」。
    },
    {
        "id": "E11",
        "question": "本來的發薪日是每個月的5號，但這個月5號剛好碰上星期六，結果公司公告說因為銀行假日不營業，所以薪水要統一延後到下星期一（7號）才撥款，這種因為假日延後發薪的行為合不合法？",
        "type": "時間規律干擾型",
        "ground_truth_pages": [51] # 誘騙重點：「銀行不營業、星期六、延後撥款」，測試系統對「發薪日遇假日之提前或延後給付規範」的掌握度。
    },
    {
        "id": "E12",
        "question": "今天是我在公司的最後一天，交接都辦完了。但我問會計今天能不能把這個月的薪水和特休折現一次結清發給我，會計卻說要等下個月統一發薪日才匯款，離職日當天依法必須立刻發錢嗎？",
        "type": "流程衝突情境型",
        "ground_truth_pages": [53] # 誘騙重點：「特休折現、匯款、交接、下個月統一發薪」，考驗模型對於「勞工離職時工資結清時限」的專一搜尋能力。
    },
    {
        "id": "E13",
        "question": "我應徵了一個兼職，一個月總薪水只有一萬五千元。不過我跟老闆談好一天只來上班四個小時、週休二日。這樣換算下來，我的月薪總額雖然遠低於兩萬多，但這樣有低於基本工資的法定標準嗎？",
        "type": "數學邏輯與部分工時型",
        "ground_truth_pages": [55] # 誘騙重點：「一萬五、低於兩萬多、四小時」，數字干擾極強，第一階段極易迷失，必須靠 Reranker 辨識「部分工時時薪與月薪折算」的核心邏輯。
    },
    {
        "id": "E14",
        "question": "我是餐飲業排班制，今天原本被排定上班六個小時。但快下班時因為店裡突然爆滿，經理叫我多留下來幫忙頂兩個小時。那我今天總共上了八小時，後面多出的那兩個小時需要給我加班費嗎？",
        "type": "正常工時變體型",
        "ground_truth_pages": [57] # 誘騙重點：總工時剛好是「八小時」，會誘騙向量模型認為「未超過法定單日八小時不需加班費」，但核心是「超過原排班工時之延長工時計算」。
    },
    {
        "id": "E15",
        "question": "老闆說為了符合周休二日，他把我們的工時改成星期五上10小時、星期六上10小時，然後星期一到星期四讓我們每天都只要來上4小時。這種透過截長補短、總工時沒超標的變形玩法是合法的嗎？",
        "type": "複雜排班干擾型",
        "ground_truth_pages": [58] # 誘騙重點：「截長補短、總工時、週休二日、10小時」，高度混淆，考驗模型對「變形工時（彈性工時）法定上限與程序」的判斷。
    },
    {
        "id": "E16",
        "question": "我在社區當大樓夜班保全，上個月因為同事請假，我狂加班導致總加班時數高達55個小時，遠超過一般勞工的46小時上限。但我去檢舉時主管卻說這可能完全合法，為什麼保全的加班上限跟別人不一樣？",
        "type": "特殊行業例外型",
        "ground_truth_pages": [61] # 誘騙重點：「46小時上限、社區大樓、檢舉」，用一般勞工標準當陷阱，考驗模型對於「勞基法第84條之1（責任制行業）特殊工時規範」的精準重排。
    },
    {
        "id": "E17",
        "question": "我們公司是全遠端自由上下班，老闆說既然大家都在家工作、沒有實體刷卡機，那就不用特別記錄大家的上下班時間了，大家互相信任就好。這種不備置出勤紀錄的說法會讓公司挨罰嗎？",
        "type": "新型態工作情境型",
        "ground_truth_pages": [60] # 誘騙重點：「遠端工作、在家工作、互信、自由上下班」，這會帶偏向量，但核心法規是「雇主備置勞工出勤紀錄之強制義務」。
    },
    {
        "id": "E18",
        "question": "今年加班累積了很多補休時數，本來想留到跨年連假一次休完，但公司人事今天發email說如果12月31號前沒有把補休用完，到了明年1月1號時數就會自動歸零失效、也不會換成錢，這樣符合勞基法嗎？",
        "type": "時間跨度陷阱型",
        "ground_truth_pages": [68] # 誘騙重點：「跨年連假、自動歸零、時數失效」，大量生活化包裝，核心考驗「補休期限屆滿未休畢應依法折算發給工資」。
    },
    {
        "id": "E19",
        "question": "我在這家公司已經不眠不休結結實實待滿整整四年了，昨天去查系統，發現我今年可以排的特休天數居然只有區區的10天。請問按照我目前累計的這份法定資歷，這個特休天數是正確的嗎？",
        "type": "數字對照精準型",
        "ground_truth_pages": [70] # 誘騙重點：出現「滿四年、10天」，第一階段若單看文字容易直接給出 True，必須依靠 Reranker 精準對齊法規中的「特休天數對照表（滿四年應為14天）」。
    },
    {
        "id": "E20",
        "question": "我是某大銀行的臨櫃櫃員，因為中午客人都爆滿來辦業務，主管叫我們所有人輪流在座位上邊吃便當邊辦公，導致中午完全沒有任何完整的空閒休息時間，銀行這樣的排班安排有沒有違反勞基法？",
        "type": "特定職業與日常行為型",
        "ground_truth_pages": [75] # 誘騙重點：「臨櫃櫃員、吃便當、辦公、中午客滿」，生活細節極多，考驗重排模型排除雜訊並鎖定「連續工作四小時應給予三十分鐘休息」的法規核心。
    }
]


In [ ]:
def evaluate_baseline_hit_at_3(evaluation_dataset, vectorstore):
    results = []

    for item in evaluation_dataset:
        question = item["question"]
        ground_truth_pages = item["ground_truth_pages"]

        # Baseline：直接 Vector Search Top 3
        docs = vectorstore.as_retriever(
            search_kwargs={"k": 3}
        ).invoke(question)

        retrieved_pages = [
            doc.metadata.get("page") for doc in docs
        ]

        # 只要 Top 3 任一頁命中 Ground Truth，就算成功
        hit = any(
            page in ground_truth_pages
            for page in retrieved_pages
        )

        results.append({
            "id": item["id"],
            "question": question,
            "type": item["type"],
            "ground_truth_pages": ground_truth_pages,
            "retrieved_pages": retrieved_pages,
            "hit@3": hit
        })

    return results

In [ ]:
baseline_results = evaluate_baseline_hit_at_3(
    evaluation_dataset,
    vectorstore
)

for result in baseline_results:
    print(
        result["id"],
        "| Ground Truth:", result["ground_truth_pages"],
        "| Retrieved:", result["retrieved_pages"],
        "| Hit:", result["hit@3"]
    )

E01 | Ground Truth: [14] | Retrieved: [14, 113, 138] | Hit: True
E02 | Ground Truth: [17] | Retrieved: [32, 11, 4] | Hit: False
E03 | Ground Truth: [21] | Retrieved: [21, 31, 25] | Hit: True
E04 | Ground Truth: [25] | Retrieved: [25, 30, 4] | Hit: True
E05 | Ground Truth: [27] | Retrieved: [27, 53, 5] | Hit: True
E06 | Ground Truth: [30] | Retrieved: [30, 32, 27] | Hit: True
E07 | Ground Truth: [32] | Retrieved: [32, 30, 11] | Hit: True
E08 | Ground Truth: [34] | Retrieved: [48, 127, 34] | Hit: True
E09 | Ground Truth: [40] | Retrieved: [40, 19, 31] | Hit: True
E10 | Ground Truth: [50] | Retrieved: [50, 40, 19] | Hit: True
E11 | Ground Truth: [51] | Retrieved: [51, 52, 53] | Hit: True
E12 | Ground Truth: [53] | Retrieved: [53, 113, 38] | Hit: True
E13 | Ground Truth: [55] | Retrieved: [55, 6, 34] | Hit: True
E14 | Ground Truth: [57] | Retrieved: [7, 58, 36] | Hit: False
E15 | Ground Truth: [58] | Retrieved: [58, 7, 85] | Hit: True
E16 | Ground Truth: [61] | Retrieved: [61, 85, 7] | Hit

In [ ]:
baseline_hit_at_3 = sum(
    result["hit@3"] for result in baseline_results
) / len(baseline_results)

print(f"Baseline Hit@3：{baseline_hit_at_3:.2%}")

Baseline Hit@3：90.00%


In [ ]:
def calculate_mrr(results):
    reciprocal_ranks = []

    for result in results:
        ground_truth_pages = result["ground_truth_pages"]
        retrieved_pages = result["retrieved_pages"]

        rr = 0

        for rank, page in enumerate(retrieved_pages, start=1):
            if page in ground_truth_pages:
                rr = 1 / rank
                break

        reciprocal_ranks.append(rr)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)


baseline_mrr = calculate_mrr(baseline_results)

print(f"Baseline MRR：{baseline_mrr:.4f}")

Baseline MRR：0.8000
